# I4. The measurement equation

The fourth notebook of the introductory track ({doc}`../index`). It writes A1
small, rang's simulated MeerKAT observation, twice: once with thermal noise
and RFI, and once without (its noiseless **twin**). Both come from the same
seeds, and the true instrument is saved next to them, so every term of the
measurement equation can be applied and removed exactly. About 30 s.

**Learning goals.** After this notebook you can

- write the measurement equation V_pq = J_p M_pq J_q^H + noise for linear feeds;
- name each Jones term in J_p and the physical effect behind it: delay K,
  bandpass B, gain G, leakage D, cross-hand phase X, parallactic rotation P;
- corrupt a sky model term by term and correct it again with the truth;
- explain why gains are determined only up to a common phase, and how a
  reference antenna fixes it.

In [ ]:
%matplotlib inline
import json
import os
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rang import core_ms
from rang import simtruth as st
from rang.notebook import data

WORK = data.default_cache_dir() / "intro-i4"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
for extra in ([], ["--noiseless", "--name", "a1-small-twin"]):
    # The A1 generator ships with rang-core: python -m rang.simtruth_ms generate small DIR
    cmd = [sys.executable, "-m", "rang.simtruth_ms", "generate", "small", str(WORK), "--workers", "2", *extra]
    out = subprocess.run(cmd, capture_output=True, text=True, check=True, env={**os.environ, "RAYON_NUM_THREADS": "2"})
    print(" ".join(["python", *cmd[1:5], "DIR", *cmd[6:]]), "->", json.loads(out.stdout.splitlines()[-1])["ms_bytes"], "bytes")
sorted(p.name for p in WORK.iterdir())

## 1. The equation

MeerKAT's feeds are linear: each antenna has an X and a Y receptor, and a
baseline (p, q) correlates them four ways, XX, XY, YX and YY. Written as a
2×2 matrix, the **coherency** of the sky for an unresolved source of Stokes
I, Q, U, V is

M = [[I + Q, U + iV], [U − iV, I − Q]].

Each antenna's signal path is a 2×2 **Jones matrix** J_p, the product of
one matrix per physical effect. The correlator measures

V_pq = J_p M_pq J_q^H + N_pq,

with N the thermal noise of I1. A1's chain, in CASA's order, is
J_p = B_p K_p Kc G_p D_p X P_p (A1 also has a target-only gain S, left out
here). Reading from the sky inwards:

| Term | Matrix | Physical effect |
| --- | --- | --- |
| P | rotation by the parallactic angle q_p | the sky turns relative to an alt-az mounted feed |
| X | diag(x(ν), 1), one for the array | the phase between the X and Y signal paths (Xf) |
| D | 1 on the diagonal, d_x, d_y off it | leakage: each receptor picks up some of the other polarisation |
| G | diag(g_x, g_y)(t) | electronic gain, amplitude and phase drifting in time |
| K, Kc | diag(exp(2πi τ (ν − ν₀))) | a delay τ in each signal path, a phase linear in frequency (Kc: the X−Y delay, KCROSS) |
| B | diag(b_x(ν), b_y(ν)) | the bandpass: the frequency response of the receivers and filters |

The truth that wrote A1 small is in `truth.npz`, raw and in CASA's gauge:

In [ ]:
d = st.design("small")
z = np.load(WORK / "truth.npz")
truth = {k.removeprefix("raw_"): z[k] for k in z.files if k.startswith("raw_")}
q = z["parallactic_angle"]  # (integration, antenna), radians
freq = z["freq_hz"]
pd.DataFrame(
    [
        {"term": "K delays", "values": f"|tau| {np.abs(truth['tau_ns']).min():.1f}-{np.abs(truth['tau_ns']).max():.1f} ns"},
        {"term": "B bandpass", "values": f"|b| {np.abs(truth['bandpass']).min():.2f}-{np.abs(truth['bandpass']).max():.2f}"},
        {"term": "G gains", "values": f"|g| {np.abs(truth['gain']).min():.2f}-{np.abs(truth['gain']).max():.2f}"},
        {"term": "D leakage", "values": f"|d| up to {np.abs(truth['leakage']).max():.3f}"},
        {"term": "X cross-hand phase", "values": f"{np.degrees(np.angle(truth['xf'])).min():.1f} to {np.degrees(np.angle(truth['xf'])).max():.1f} deg"},
        {"term": "P parallactic angle", "values": f"{np.degrees(q).min():.1f} to {np.degrees(q).max():.1f} deg"},
    ]
)

## 2. The equation reproduces the data

Read the twin's DATA on the flux calibrator J1939−6342, build M from its
model (I1: Stevens-Reynolds 2016, unpolarised) and apply the true chain with
`rang.simtruth.corrupt`:

In [ ]:
with core_ms.open_table(str(WORK / "a1-small-twin.ms")) as t:
    a1, a2, field = t.getcol("ANTENNA1"), t.getcol("ANTENNA2"), t.getcol("FIELD_ID")
    twin = t.getcol("DATA")
with core_ms.open_table(str(WORK / "a1-small.ms")) as t:
    noisy = t.getcol("DATA")
nbl = len(d.antennas) * (len(d.antennas) - 1) // 2
tindex = np.arange(len(a1)) // nbl  # rows are written integration by integration
fcal = field == 0
rows = st.Rows(tindex[fcal], a1[fcal], a2[fcal])
stokes = st.calibrator_stokes(d)[0]  # (channel, I Q U V) of J1939-6342
model = np.repeat(st.stokes_coherency(stokes)[None], fcal.sum(), axis=0)
predicted = st.corrupt(model, truth, d, rows, q)
print(f"{fcal.sum()} rows; max |J M J^H - DATA(twin)| = {np.abs(predicted - twin[fcal]).max():.1e} Jy (DATA is stored in single precision)")

The noise term is the difference between the two sets. Outside the cells
hit by RFI (I5) it has exactly the radiometer σ of I1:

In [ ]:
rfi = np.unpackbits(np.load(WORK / "a1-small.rfi_mask.npz")["any"], axis=1, count=d.nchan).astype(bool)
noise = (noisy - twin)[~rfi]
print(f"std of the real and imaginary parts: {noise.real.std():.4f}, {noise.imag.std():.4f} Jy; design sigma {d.sigma:.4f} Jy")

## 3. One term at a time

Switch every term off (identity), then on again one by one, and watch a
baseline's XX visibility on the flux calibrator change:

In [ ]:
identity = {
    "tau_ns": 0 * truth["tau_ns"],
    "k": np.ones_like(truth["k"]),
    "bandpass": np.ones_like(truth["bandpass"]),
    "gain": np.ones_like(truth["gain"]),
    "selfcal": np.ones_like(truth["selfcal"]),
    "kcross": np.ones_like(truth["kcross"]),
    "xf": np.ones_like(truth["xf"]),
    "leakage": np.zeros_like(truth["leakage"]),
}
steps = [("sky (M)", []), ("+ K", ["k", "kcross"]), ("+ B", ["bandpass"]), ("+ G", ["gain"])]
row = np.flatnonzero((rows.a1 == 1) & (rows.a2 == 5))[0]
one = st.Rows(rows.tindex[[row]], rows.a1[[row]], rows.a2[[row]])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
on = dict(identity)
for label, keys in steps:
    for k in keys:
        on[k] = truth[k]
    v = st.corrupt(model[[row]], on, d, one, np.full_like(q, -st.RECEPTOR_ANGLE))[0, :, 0]
    axes[0].plot(freq / 1e6, np.abs(v), label=label)
    axes[1].plot(freq / 1e6, np.degrees(np.angle(v)), label=label)
axes[0].set(xlabel="frequency (MHz)", ylabel="|XX| (Jy)", title=f"m{4 * one.a1[0]:03d}-m{4 * one.a2[0]:03d}, XX amplitude")
axes[1].set(xlabel="frequency (MHz)", ylabel="phase (deg)", title="XX phase")
axes[0].legend()
plt.show()

K tilts the phase linearly in frequency (the delays I2 measured on real
data), B shapes amplitude and phase across the band, G scales and rotates
the whole spectrum by a time-dependent complex number. (P is held at zero
rotation here: `-RECEPTOR_ANGLE` cancels the feeds' fixed −90°.)

The polarisation terms show on a polarised source. A1's polarisation
calibrator 3C286-SIM is linearly polarised. As the parallactic angle turns
over the observation, P moves that polarisation between the correlations; D
and X add instrumental cross-hand signal:

In [ ]:
print(f"3C286-SIM: polarisation fraction {st.XCAL['p']:.3f}, angle {st.XCAL['chi_deg']:.0f} deg, rotation measure {st.XCAL['rm']:.0f} rad/m^2")
xcal = field == 1
rx = st.Rows(tindex[xcal], a1[xcal], a2[xcal])
mx = np.repeat(st.stokes_coherency(st.calibrator_stokes(d)[1])[None], xcal.sum(), axis=0)
only_p = st.corrupt(mx, identity, d, rx, q)
with_dx = st.corrupt(mx, {**identity, "leakage": truth["leakage"], "xf": truth["xf"]}, d, rx, q)
centre = len(freq) // 2
bl = (rx.a1 == 0) & (rx.a2 == 1)
pa = np.degrees(q[rx.tindex[bl], 0])
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(pa, only_p[bl, centre, 1].real, "o", label="P only: Re XY")
ax.plot(pa, only_p[bl, centre, 1].imag, "o", mfc="none", label="P only: Im XY")
ax.plot(pa, with_dx[bl, centre, 1].real, "x", label="P, X and D: Re XY")
ax.plot(pa, with_dx[bl, centre, 1].imag, "+", label="P, X and D: Im XY")
ax.set(xlabel="parallactic angle q (deg)", ylabel="XY (Jy)", title=f"3C286-SIM, m000-m004, {freq[centre] / 1e6:.0f} MHz")
ax.legend()
plt.show()

Notebook I8 uses exactly this rotation to tell the source's polarisation
from the instrument's.

## 4. Correcting with the truth

`rang.simtruth.correct` inverts the chain. Applied to the twin it returns the
sky to rounding; applied to the noisy data it returns the sky plus noise
(and RFI) passed through the inverse gains:

In [ ]:
back = st.correct(twin[fcal], truth, d, rows, q)
print(f"twin: max |corrected - M| = {np.abs(back - model).max():.1e} Jy")
clean = ~rfi[fcal]
resid = (st.correct(noisy[fcal], truth, d, rows, q) - model)[..., [0, 3]][clean]
print(f"noisy, RFI-free cells, XX and YY: std of corrected - M {resid.real.std():.4f} Jy (raw sigma {d.sigma:.4f} Jy, scaled by 1/|g_p g_q|)")

Calibration (I6) has to find these terms without knowing them, from
calibrator observations whose sky M is known.

## 5. A phase nobody can measure

Multiply every antenna's gain by the same phase factor e^{iφ}. Each
visibility gets e^{iφ} e^{−iφ} = 1: the data cannot tell the two gain sets
apart.

In [ ]:
phi = np.exp(1j * 0.7)
rotated = {**truth, "gain": truth["gain"] * phi}
diff = np.abs(st.corrupt(model, rotated, d, rows, q) - predicted).max()
print(f"max |V(rotated gains) - V(true gains)| = {diff:.1e} Jy")

A solver therefore fixes the phase of one **reference antenna** to zero (per
hand) and expresses every other antenna relative to it. `casa_gauge` puts
the truth in CASA's convention for reference antenna m000, the form the
calibration tables of I6 are compared against:

In [ ]:
gauge = st.casa_gauge(truth, 0, freq, float(z["nu0_hz"]))
print(f"refant gain phase, raw: up to {np.degrees(np.abs(np.angle(truth['gain'][:, 0]))).max():.1f} deg; in the gauge: {np.degrees(np.abs(np.angle(gauge['gain'][:, 0]))).max():.1e} deg")
print(f"refant delay in the gauge: {gauge['tau_ns'][0]} ns; matches the stored truth: {np.allclose(gauge['tau_ns'], z['gauge0_tau_ns'])}")

The common phase is only the simplest ambiguity. With full 2×2 Jones
matrices any unitary matrix can be moved between antennas (the U(2) gauge),
which leaves 8n − 4 real degrees of freedom for n antennas: notebook 01
counts them.

## Exercises

1. Turn on only D (leakage) on the unpolarised flux calibrator. Which
   correlations change, and by how much relative to I?
2. Use the noisy data and the truth to measure σ separately on XX and XY.
   Should they differ?
3. Add a 30° error to the parallactic angle passed to `correct` for 3C286-SIM.
   How large is the leftover in XY and YX?
4. Pick reference antenna m020 (index 5) in `casa_gauge`. Which terms change,
   and which visibilities?

## Where next

- {doc}`i5-rfi-and-flagging`: the RFI that the noise check above stepped
  around.
- Notebook {doc}`01-data-and-rank` and {doc}`../../concepts/gains`: rank,
  8n − 4 and the U(2) gauge.
- {doc}`i2-visibilities` for what a visibility is.